# General Data Visualization

Interactive overview of the main characteristics of the Shelter Animal Outcomes dataset.

In [43]:
import numpy as np
import pandas as pd
import plotly.express as px

In [44]:
from configurations import OUTPUT_DIR, TARGET, TRAIN_PATH

# Output filename for the outcome distribution figure.
OUTCOME_DISTRIBUTION_FILENAME = "visualization_outcome_distribution.png"
# Output filename for the monthly outcomes figure.
MONTHLY_OUTCOMES_FILENAME = "visualization_monthly_outcomes.png"
# Output filename for the animal type outcomes figure.
ANIMAL_TYPE_OUTCOMES_FILENAME = "visualization_animal_type_outcomes.png"
# Output filename for the age distribution figure.
AGE_DISTRIBUTION_FILENAME = "visualization_age_distribution.png"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train = pd.read_csv(TRAIN_PATH)
train["DateTime"] = pd.to_datetime(train["DateTime"], errors="coerce")
train["YearMonth"] = train["DateTime"].dt.to_period("M").astype(str)

In [45]:
import importlib
import plotting

importlib.reload(plotting)
from plotting import show_and_save

In [46]:
def parse_age_in_months(value):
    """Convert an age string to months."""
    if pd.isna(value):
        return np.nan
    amount, unit = str(value).split(maxsplit=1)
    multiplier = {"day": 1 / 30, "days": 1 / 30, "week": 1 / 4.345, "weeks": 1 / 4.345,
                  "month": 1, "months": 1, "year": 12, "years": 12}
    return float(amount) * multiplier[unit.lower()]


def finish_dataset(frame):
    """Prepare date and age variables for visualization."""
    result = frame.copy()
    result["AgeMonths"] = result["AgeuponOutcome"].map(parse_age_in_months)
    result["YearMonth"] = result["DateTime"].dt.to_period("M").astype(str)
    return result


train = finish_dataset(train)

In [47]:
outcome_counts = train[TARGET].value_counts().rename_axis(TARGET).reset_index(name="count")
fig = px.bar(
    outcome_counts,
    x=TARGET,
    y="count",
    color=TARGET,
    text="count",
    labels={TARGET: "Outcome", "count": "Count"},
    title="Outcome type distribution",
)
show_and_save(fig, OUTCOME_DISTRIBUTION_FILENAME)

Figure saved: output/visualization_outcome_distribution.png


PosixPath('output/visualization_outcome_distribution.png')

In [48]:
monthly = train.groupby(["YearMonth", TARGET], as_index=False).size()
fig = px.line(
    monthly,
    x="YearMonth",
    y="size",
    color=TARGET,
    markers=True,
    labels={"YearMonth": "Year-month", "size": "Count", TARGET: "Outcome"},
    title="Monthly outcomes",
)
fig.update_xaxes(type="category", tickangle=45)
show_and_save(fig, MONTHLY_OUTCOMES_FILENAME)

Figure saved: output/visualization_monthly_outcomes.png


PosixPath('output/visualization_monthly_outcomes.png')

In [49]:
type_outcome = train.groupby(["AnimalType", TARGET], as_index=False).size()
fig = px.bar(
    type_outcome,
    x="AnimalType",
    y="size",
    color=TARGET,
    barmode="group",
    labels={"AnimalType": "Animal type", "size": "Count", TARGET: "Outcome"},
    title="Outcomes by animal type",
)
show_and_save(fig, ANIMAL_TYPE_OUTCOMES_FILENAME)

Figure saved: output/visualization_animal_type_outcomes.png


PosixPath('output/visualization_animal_type_outcomes.png')

In [50]:
fig = px.histogram(
    train.dropna(subset=["AgeMonths"]),
    x="AgeMonths",
    color="AnimalType",
    marginal="box",
    nbins=35,
    labels={"AgeMonths": "Age (months)", "count": "Count", "AnimalType": "Animal type"},
    title="Age distribution by animal type",
)
show_and_save(fig, AGE_DISTRIBUTION_FILENAME)

Figure saved: output/visualization_age_distribution.png


PosixPath('output/visualization_age_distribution.png')